# SAFE4ALL Climate Risk Assessment: Floods and Drought

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nelson1798/NOAA-Safe4All/blob/main/SAFE4ALL_Climate_Risk_Assessment_Workshop.ipynb)

## Kenya · Zimbabwe · Ghana

This is a single, guided workshop notebook. Run it from top to bottom (**Runtime -> Run all** in Colab, or **Run -> Run All Cells** in Jupyter/Anaconda). It first retrieves and quality-checks TAHMO observations, then uses them to interpret TAMSAT, CHIRPS, and IMERG rainfall products before producing screening-level flood and drought assessments.

**Important:** outputs are climate-risk screening products—not operational forecasts or warnings. Combine them with local hydrology, terrain, exposure, vulnerability, and impact information before making decisions.

## 0. Environment setup (run once)

This cell makes the notebook self-contained on **Colab**, local **Jupyter**, and **Anaconda** alike — no separate setup step is required:

- On Colab (opened straight from GitHub via the badge above), only this notebook file is fetched. The cell clones the rest of the project (the `safe4all` package) into the runtime and adds it to `sys.path`.
- On a local Jupyter/Anaconda install that already has this project cloned (see the README), `safe4all` is already importable, so cloning is skipped.
- Any Python package the notebook needs that is not already installed is installed quietly with `pip`. This is a fast no-op if you created the `environment.yml` conda environment first, and it is what makes plain "Run all" work even without it.

In [ ]:
# --- Environment setup --------------------------------------------------------
# Safe to re-run. No separate install step or config file is required.
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Nelson1798/NOAA-Safe4All.git'
REPO_BRANCH = 'main'


def _in_colab() -> bool:
    return importlib.util.find_spec('google.colab') is not None


def _find_spec(module_name: str):
    try:
        return importlib.util.find_spec(module_name)
    except ModuleNotFoundError:
        # Raised for a dotted name whose parent package is itself missing.
        return None


if _in_colab() and _find_spec('safe4all') is None:
    # The "Open in Colab" badge only fetches this .ipynb file, so the rest of
    # the project (the safe4all package) is cloned into the runtime here.
    repo_path = Path('/content/NOAA-Safe4All')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(repo_path)],
            check=True,
        )
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

# Packages the notebook needs that are not part of a stock Colab/Jupyter
# install. Installing is a quick no-op for anything already present (for
# example after `conda env create -f environment.yml`).
REQUIRED_PACKAGES = [
    ('ee', 'earthengine-api'),
    ('geemap', 'geemap'),
    ('matplotlib', 'matplotlib'),
    ('xarray', 'xarray'),
    ('netCDF4', 'netCDF4'),
]
missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES if _find_spec(module_name) is None]
if missing:
    print(f"Installing missing packages: {', '.join(missing)}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import safe4all  # noqa: F401  (fails loudly here if the package still can't be found)

print('Environment ready:', Path(safe4all.__file__).parent)

## 1. Configure the exercise

Change only the variables in the next cell. The same country and dates are used throughout the notebook so that station validation, flood analysis, and drought analysis remain comparable.

In [ ]:
# --- Workshop choices --------------------------------------------------------
COUNTRY = 'Ghana'  # Choose: Kenya, Zimbabwe, Ghana
ANALYSIS_START = '2024-03-01'
ANALYSIS_END = '2024-05-31'
EXTREME_RAIN_THRESHOLD_MM = 50
BASELINE_START_YEAR = 1991
BASELINE_END_YEAR = 2020
MAX_STATIONS = 10  # Start small for a workshop; increase after testing.

# Google Cloud project with the Earth Engine API enabled. Required by Earth
# Engine for every request. Pre-set to the SAFE4ALL workshop project; change
# it if you are running this against your own Cloud project instead.
EE_PROJECT = 'natural-notch-435413-j3'

# Optional: local path to a service-account JSON key for headless Earth
# Engine auth (recommended for a shared workshop project so participants
# don't each need their own OAuth login). Leave as None to use the normal
# interactive ee.Authenticate() prompt instead.
#
# IMPORTANT: never commit a key file to this repo. Keep it outside the
# project folder, or inside .safe4all-cache/ (already git-ignored), and
# point this at its local path -- or just set the EE_SERVICE_ACCOUNT_KEY /
# GOOGLE_APPLICATION_CREDENTIALS environment variable before starting
# Jupyter instead of editing this cell.
EE_SERVICE_ACCOUNT_KEY = None

# Exact Google Shared Drive name. Set this before the workshop if different.
SHARED_DRIVE_NAME = 'SAFE4ALL-workshop'

# Optional: path to an approved TAMSAT NetCDF subset for the first live run.
# It will be copied to the Shared Drive cache and reused by later runs.
TAMSAT_SOURCE_FILE = None

import os
os.environ['SAFE4ALL_SHARED_DRIVE'] = SHARED_DRIVE_NAME
os.environ['EE_PROJECT'] = EE_PROJECT
if EE_SERVICE_ACCOUNT_KEY:
    os.environ['EE_SERVICE_ACCOUNT_KEY'] = EE_SERVICE_ACCOUNT_KEY

from safe4all.config import country_settings
settings = country_settings(COUNTRY)
settings


## 2. Data access and caching

The first successful live pull is cached. In Colab, the notebook mounts the named Shared Drive; if it is unavailable, it uses a SAFE4ALL folder in MyDrive. In local Jupyter it uses `.safe4all-cache`. TAHMO credentials are requested privately at runtime and are never saved in this project or the cache.

In [ ]:
import getpass
from safe4all.cache import cache_root

if not os.environ.get('TAHMO_API_KEY'):
    os.environ['TAHMO_API_KEY'] = getpass.getpass('TAHMO API key: ')
if not os.environ.get('TAHMO_API_SECRET'):
    os.environ['TAHMO_API_SECRET'] = getpass.getpass('TAHMO API secret: ')

CACHE_ROOT = cache_root()
print(f'Workshop cache: {CACHE_ROOT}')

## 3. Retrieve and quality-check TAHMO station rainfall

TAHMO stations are the local reference. The notebook retains quality diagnostics and flags implausible daily totals instead of silently deleting them. If a country has few stations, that limitation is itself an important workshop finding.

In [ ]:
import pandas as pd
from safe4all.cache import cached_file
from safe4all.qc import qc_daily_precipitation
from safe4all.tahmo import save_station_metadata, station_precipitation

try:
    metadata_path = cached_file(
        f'tahmo/{COUNTRY}/station_metadata.csv',
        lambda path: save_station_metadata(path, COUNTRY),
    )
    stations = pd.read_csv(metadata_path).head(MAX_STATIONS)
    display(stations[['code', 'location.latitude', 'location.longitude']])
except Exception as error:
    stations = pd.DataFrame(columns=['code', 'location.latitude', 'location.longitude'])
    print(f'Could not retrieve TAHMO station metadata: {error}')
    print('Continue with satellite products, but document this coverage limitation.')

station_frames = []
for code in stations['code']:
    try:
        path = cached_file(
            f'tahmo/{COUNTRY}/{ANALYSIS_START}_{ANALYSIS_END}/{code}_precip.csv',
            lambda target, code=code: station_precipitation(code, ANALYSIS_START, ANALYSIS_END).to_csv(target, index=False),
        )
        frame = pd.read_csv(path)
        if not frame.empty:
            station_frames.append(frame)
    except Exception as error:
        print(f'Could not retrieve {code}: {error}')

if station_frames:
    tahmo_daily = qc_daily_precipitation(pd.concat(station_frames, ignore_index=True))
    qc_summary = tahmo_daily.groupby('station', as_index=False).agg(
        days=('date', 'nunique'), usable_days=('usable', 'sum'),
        negative_days=('negative_flag', 'sum'), extreme_days=('extreme_flag', 'sum'),
        total_rainfall_mm=('precipitation_mm', 'sum'),
    )
    qc_summary['usable_fraction'] = qc_summary['usable_days'] / qc_summary['days']
    display(qc_summary.sort_values('usable_fraction'))
else:
    tahmo_daily = pd.DataFrame()
    qc_summary = pd.DataFrame()
    print('No station observations were returned. Continue with satellite products, but document this coverage limitation.')

## 4. Load Google Earth Engine products

CHIRPS provides an independent long-term rainfall comparison. IMERG provides event-scale, half-hourly rainfall context for flood cases. Earth Engine access is interactive on first use; complete the authentication prompt, then rerun this cell. Make sure `EE_PROJECT` in Section 1 is set to a Google Cloud project with the Earth Engine API enabled — Earth Engine requires this for every request.

In [ ]:
try:
    import geemap
    from safe4all.gee import country_boundary, initialise_earth_engine

    ee = initialise_earth_engine()
    boundary = country_boundary(ee, COUNTRY)
    Map = geemap.Map(center=settings['center'], zoom=settings['zoom'])
    Map.addLayer(boundary.style(color='black', fillColor='00000000', width=2), {}, f'{COUNTRY} boundary')
    Map.addLayerControl()
    display(Map)
except Exception as error:
    ee = None
    boundary = None
    print(f'Earth Engine is not ready: {error}')

## 5. Validate gridded rainfall against TAHMO

This comparison uses period totals at station locations. Assess bias and disagreement before relying on a national map. TAMSAT should be sampled from the cached NetCDF subset in the next section; CHIRPS and IMERG are sampled here directly through Earth Engine.

In [ ]:
from safe4all.risk import chirps_daily, imerg_event_rainfall
from safe4all.validation import rainfall_metrics

comparison = pd.DataFrame()
if ee is not None and not tahmo_daily.empty:
    valid_codes = qc_summary.loc[qc_summary['usable_fraction'].ge(0.8), 'station']
    point_features = [
        ee.Feature(ee.Geometry.Point([row['location.longitude'], row['location.latitude']]), {'station': row['code']})
        for _, row in stations[stations['code'].isin(valid_codes)].iterrows()
    ]
    if point_features:
        points = ee.FeatureCollection(point_features)
        region = points.geometry().buffer(50000)
        chirps_total = chirps_daily(ee, ANALYSIS_START, ANALYSIS_END, region).sum().rename('chirps_mm')
        imerg_total = imerg_event_rainfall(ee, ANALYSIS_START, ANALYSIS_END, region).rename('imerg_mm')
        reducer = ee.Reducer.first().setOutputs(['chirps_mm', 'imerg_mm'])
        satellite_df = geemap.ee_to_df(chirps_total.addBands(imerg_total).reduceRegions(points, reducer, 5500))
        ground_total = tahmo_daily[tahmo_daily['usable']].groupby('station', as_index=False)['precipitation_mm'].sum()
        comparison = ground_total.merge(satellite_df[['station', 'chirps_mm', 'imerg_mm']], on='station', how='inner')
        display(comparison)
        for product in ('chirps_mm', 'imerg_mm'):
            if len(comparison) >= 3:
                print(product, rainfall_metrics(comparison['precipitation_mm'], comparison[product]))
    else:
        print('No stations met the minimum QC coverage threshold.')
else:
    print('Validation will run after both TAHMO and Earth Engine are available.')

## 6. TAMSAT drought context

TAMSAT is the primary drought-monitoring product. For the first run, download an approved TAMSAT rainfall and, where available, root-zone soil-moisture subset from the official data-access service. Set `TAMSAT_SOURCE_FILE` in Section 1. The notebook copies it to the Shared Drive cache, so later users do not download it again. Use `rfe` for validation where available; use `rfe_filled` only for complete time-series analysis.

In [ ]:
import matplotlib.pyplot as plt
from safe4all.cache import copy_to_cache

tamsat_ds = None
tamsat_cache = CACHE_ROOT / f'tamsat/{COUNTRY}/tamsat_subset.nc'
if TAMSAT_SOURCE_FILE:
    tamsat_cache = copy_to_cache(TAMSAT_SOURCE_FILE, f'tamsat/{COUNTRY}/tamsat_subset.nc')
if tamsat_cache.exists():
    import xarray as xr
    tamsat_ds = xr.open_dataset(tamsat_cache)
    rainfall_var = 'rfe' if 'rfe' in tamsat_ds.data_vars else 'rfe_filled'
    tamsat_rainfall = tamsat_ds[rainfall_var].sel(time=slice(ANALYSIS_START, ANALYSIS_END))
    lon_min, lat_min, lon_max, lat_max = settings['bbox']
    lon_name = 'lon' if 'lon' in tamsat_rainfall.coords else 'longitude'
    lat_name = 'lat' if 'lat' in tamsat_rainfall.coords else 'latitude'
    lat_values = tamsat_rainfall[lat_name]
    lat_slice = slice(lat_min, lat_max) if float(lat_values[0]) < float(lat_values[-1]) else slice(lat_max, lat_min)
    country_tamsat = tamsat_rainfall.sel({lon_name: slice(lon_min, lon_max), lat_name: lat_slice})
    country_tamsat.sum('time').plot(figsize=(10, 6), cmap='YlGnBu', robust=True)
    plt.title(f'TAMSAT rainfall total: {COUNTRY}, {ANALYSIS_START} to {ANALYSIS_END}')
    plt.show()
else:
    print('No cached TAMSAT subset yet. Set TAMSAT_SOURCE_FILE for the first run; the rest of the notebook remains usable.')

## 7. Flood-risk screening

Flood risk is more than rainfall. This section maps rainfall-driven hazard only: seasonal rainfall, heavy-rainfall days, and IMERG event totals. Use it to identify locations where river networks, slope, drainage, settlements, roads, population, and impact records should be examined next.

In [ ]:
if ee is not None:
    rainfall = chirps_daily(ee, ANALYSIS_START, ANALYSIS_END, boundary)
    seasonal_rainfall = rainfall.sum().rename('seasonal_rainfall_mm').clip(boundary)
    extreme_days = rainfall.map(lambda image: image.gte(EXTREME_RAIN_THRESHOLD_MM)).sum().rename('extreme_rain_days').clip(boundary)
    imerg_event = imerg_event_rainfall(ee, ANALYSIS_START, ANALYSIS_END, boundary).clip(boundary)
    rainfall_hazard = extreme_days.unitScale(0, 10).clamp(0, 1).rename('relative_rainfall_hazard')

    FloodMap = geemap.Map(center=settings['center'], zoom=settings['zoom'])
    FloodMap.addLayer(seasonal_rainfall, {'min': 0, 'max': 1000, 'palette': ['fff7bc', 'fec44f', 'd95f0e', '993404']}, 'CHIRPS seasonal rainfall (mm)')
    FloodMap.addLayer(extreme_days, {'min': 0, 'max': 10, 'palette': ['ffffcc', 'a1dab4', '41b6c4', '225ea8']}, f'CHIRPS days ≥ {EXTREME_RAIN_THRESHOLD_MM} mm')
    FloodMap.addLayer(imerg_event, {'min': 0, 'max': 1000, 'palette': ['ffffcc', 'a1dab4', '41b6c4', '225ea8']}, 'IMERG event rainfall (mm)')
    FloodMap.addLayer(rainfall_hazard, {'min': 0, 'max': 1, 'palette': ['ffffb2', 'fecc5c', 'fd8d3c', 'e31a1c']}, 'Relative rainfall flood hazard')
    FloodMap.addLayer(boundary.style(color='black', fillColor='00000000', width=2), {}, 'Country boundary')
    FloodMap.addLayerControl()
    display(FloodMap)
else:
    print('Authenticate Earth Engine to map flood hazard.')

## 8. Drought-risk screening

This section maps a standardized CHIRPS seasonal rainfall anomaly against 1991–2020. Interpret negative anomalies alongside TAMSAT rainfall, TAMSAT root-zone soil moisture, TAHMO observations, crop calendars, and reported impacts. A rainfall anomaly alone is not a drought-impact assessment.

In [ ]:
if ee is not None:
    season_start_month = int(ANALYSIS_START[5:7])
    season_end_month = int(ANALYSIS_END[5:7])
    analysis_year = int(ANALYSIS_START[:4])

    def seasonal_total(year):
        start = ee.Date.fromYMD(year, season_start_month, 1)
        end = ee.Date.fromYMD(year, season_end_month, 1).advance(1, 'month')
        return chirps_daily(ee, start, end, boundary).sum().rename('seasonal_rainfall').set('year', year)

    current = seasonal_total(analysis_year).clip(boundary)
    baseline = ee.ImageCollection.fromImages(ee.List.sequence(BASELINE_START_YEAR, BASELINE_END_YEAR).map(seasonal_total))
    baseline_mean = baseline.mean()
    baseline_std = baseline.reduce(ee.Reducer.stdDev())
    anomaly = current.subtract(baseline_mean).divide(baseline_std.max(1)).rename('standardized_rainfall_anomaly')
    drought_intensity = anomaly.multiply(-1).rename('relative_drought_intensity')

    DroughtMap = geemap.Map(center=settings['center'], zoom=settings['zoom'])
    DroughtMap.addLayer(anomaly, {'min': -2.5, 'max': 2.5, 'palette': ['8c510a', 'd8b365', 'f5f5f5', '5ab4ac', '01665e']}, 'CHIRPS standardized anomaly')
    DroughtMap.addLayer(drought_intensity.updateMask(drought_intensity.gt(0)), {'min': 0, 'max': 2.5, 'palette': ['fff7bc', 'fec44f', 'd95f0e', 'b30000']}, 'Relative drought intensity')
    DroughtMap.addLayer(boundary.style(color='black', fillColor='00000000', width=2), {}, 'Country boundary')
    DroughtMap.addLayerControl()
    display(DroughtMap)
else:
    print('Authenticate Earth Engine to map drought screening indicators.')

## 9. Interpretation and next steps

1. Review station coverage and QC before trusting the maps.
2. Compare TAHMO totals with CHIRPS and IMERG; document bias and disagreement.
3. Use TAMSAT rainfall and root-zone soil moisture as the core drought narrative.
4. Treat flood and drought outputs as hazard screening only. Add rivers, slope, drainage, land cover, population, infrastructure, livelihoods, and observed impacts for a complete risk assessment.
5. Keep the cached datasets and a short data-log so participants can reproduce the workshop without repeated API calls.